## 1 — Privacy-Gated Routing: PPHD Feature Hashing + ε-LDP → Student Stress Classifier → Slots Payload (No Raw Text)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
PROJECT_DIR = "/content/drive/MyDrive/pphd_agent_demo"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CKPT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

CSV_PATH = os.path.join(DATA_DIR, "dreaddit.csv")
CKPT_PATH = os.path.join(CKPT_DIR, "student_demo_D75_eps1.pt")

print("CSV_PATH:", CSV_PATH)
print("CKPT_PATH:", CKPT_PATH)
print("data dir:", os.listdir(DATA_DIR)[:10])

CSV_PATH: /content/drive/MyDrive/pphd_agent_demo/data/dreaddit.csv
CKPT_PATH: /content/drive/MyDrive/pphd_agent_demo/checkpoints/student_demo_D75_eps1.pt
data dir: ['dreaddit.csv']


In [3]:
import math
import random
import hashlib
import json
from dataclasses import dataclass
from typing import List, Tuple, Optional

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

In [4]:
# -----------------------------
# Config (PPHD Router Demo)
# -----------------------------
@dataclass
class CFG:
    # Data
    csv_path: str = CSV_PATH
    text_col: str = "text"
    label_col: Optional[str] = None  # Auto-detect a binary label column if not specified

    # Privacy-preserving feature space
    d: int = 75           # Hashing dimension (D). Smaller D => more collisions, stronger obfuscation
    epsilon: float = 1.0  # LDP privacy budget (higher => less noise, lower => more noise)
    seed: int = 0         # Seed for reproducibility (hashing + randomized response)

    # Student model size (demo version; no teacher alignment needed)
    emb_dim: int = 128    # Latent embedding size inside the student model (free choice for Day1 demo)
    hidden: int = 128
    dropout: float = 0.3

    # Training hyperparameters
    epochs: int = 5
    batch_size: int = 256
    lr: float = 1e-3

    # Runtime
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    out_path: str = CKPT_PATH  # Where to save the best student checkpoint

cfg = CFG()
print("device:", cfg.device)


# -----------------------------
# Tokenization (lightweight)
# -----------------------------
def simple_tokenize(text: str) -> List[str]:
    """
    Minimal tokenizer for the demo:
    - unigrams: individual words
    - bigrams: word_i + '_' + word_{i+1}
    """
    words = text.lower().split()
    return words + [words[i] + "_" + words[i + 1] for i in range(len(words) - 1)]


# -----------------------------
# Stable hashing utilities
# -----------------------------
def _stable_hash_to_int(s: str, seed: int = 0) -> int:
    """
    Stable hash function (unlike Python built-in hash()).
    This ensures consistent bucket/sign assignments across runs and machines.
    """
    h = hashlib.blake2b(digest_size=8, person=str(seed).encode("utf-8"))
    h.update(s.encode("utf-8"))
    return int.from_bytes(h.digest(), "big", signed=False)


# -----------------------------
# PPHD-style privatized hashed vector (D-dim)
# -----------------------------
def hashed_vector(text: str, D: int, epsilon: float, seed: int) -> List[int]:
    """
    Convert raw text into a D-dimensional signed hashed vector, then apply a lightweight ε-LDP step.

    Steps:
    1) Signed feature hashing:
       - bucket = stable_hash(token) % D
       - sign   = +/-1 from a second stable hash
       - accumulate signed counts into h[bucket]

    2) ε-LDP (randomized response, sign-flip):
       - With probability p_keep, keep the sign of each dimension
       - Otherwise, flip the sign
       - p_keep = exp(eps) / (exp(eps) + 1)

    Output:
    - A length-D list of integers (privatized representation)
    - Raw text is not retained in the representation (non-invertible in practice)
    """
    h = [0] * D
    tokens = simple_tokenize(text)

    for tok in tokens:
        bucket = _stable_hash_to_int(tok, seed) % D
        sign_bit = _stable_hash_to_int("sign_" + tok, seed) % 2
        sign = -1 if sign_bit == 1 else 1
        h[bucket] += sign

    # ε-LDP sign-flip randomized response
    rng = random.Random(seed)
    p_keep = math.exp(epsilon) / (math.exp(epsilon) + 1.0)
    return [v if rng.random() < p_keep else -v for v in h]


def build_hashed_features(texts: List[str]) -> torch.Tensor:
    """
    Batch version: convert a list of texts into an (N, D) float tensor
    for student model training/inference.
    """
    X = [hashed_vector(t, cfg.d, cfg.epsilon, cfg.seed) for t in texts]
    return torch.tensor(X, dtype=torch.float32)

device: cpu


In [5]:
# -----------------------------
# Label column auto-detection
# -----------------------------
def detect_label_col(df: pd.DataFrame, text_col: str) -> str:
    """
    Heuristically detect a binary label column.

    Priority:
    1) Common label column names (label/target/etc.)
    2) Fallback: any numeric column (excluding text_col) with exactly 2 unique values

    Raises:
        ValueError if no suitable binary label column is found.
    """
    # Common label column names
    for c in ["label", "labels", "y", "target", "stress", "class"]:
        if c in df.columns:
            return c

    # Fallback: find a binary numeric column
    for c in df.columns:
        if c == text_col:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            uniq = df[c].dropna().unique()
            if len(uniq) == 2:
                return c

    raise ValueError(
        f"Could not auto-detect binary label column. Columns={list(df.columns)}"
    )


# -----------------------------
# Student model (lightweight router)
# -----------------------------
class StudentModel(nn.Module):
    """
    Lightweight student classifier used for privacy-gated routing.

    Input:
        x: (batch_size, dim) privatized hashed features (dim = D)
    Output:
        logits: (batch_size, num_labels) class scores
        emb: (batch_size, emb_dim) intermediate embedding (useful for future KD/COS alignment)
    """
    def __init__(self, dim: int, emb_dim: int, hidden: int, dropout: float, num_labels: int = 2):
        super().__init__()

        # Encoder: project D-dim hashed features into a dense embedding space
        self.encoder = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, emb_dim),
            nn.LayerNorm(emb_dim)
        )

        # Classifier head: map embedding to logits (binary by default)
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Linear(emb_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_labels)
        )

    def forward(self, x):
        emb = self.encoder(x)
        logits = self.classifier(emb)
        return logits, emb


# -----------------------------
# Evaluation (F1 + ROC-AUC)
# -----------------------------
@torch.no_grad()
def evaluate(model, X, y):
    """
    Evaluate the student on a validation set.

    - Uses softmax(logits)[:, 1] as the positive-class probability (stress=1).
    - Converts probabilities to hard labels with a default threshold of 0.5.
    - Returns:
        (f1_score, roc_auc)
    """
    model.eval()
    logits, _ = model(X)

    # Positive-class probability (label=1)
    prob = F.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()

    # Default classification threshold (can be tuned later)
    pred = (prob >= 0.5).astype(int)

    y_np = y.detach().cpu().numpy()
    return f1_score(y_np, pred), roc_auc_score(y_np, prob)

In [6]:
# -----------------------------
# Reproducibility
# -----------------------------
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

# -----------------------------
# Load dataset (Dreaddit)
# -----------------------------
df = pd.read_csv(cfg.csv_path)
assert cfg.text_col in df.columns, df.columns  # Fail fast if the text column is missing

# Auto-detect the binary label column if not explicitly provided
label_col = cfg.label_col or detect_label_col(df, cfg.text_col)
print("Using label_col:", label_col)

# Keep only text + label, drop missing values, and ensure text is string type
df = df[[cfg.text_col, label_col]].dropna()
df[cfg.text_col] = df[cfg.text_col].astype(str)

# -----------------------------
# Train/validation split
# -----------------------------
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=cfg.seed,
    stratify=df[label_col]  # Preserve class balance across splits
)

# -----------------------------
# Build privacy-preserving features (PPHD-style)
# Raw text -> hashed signed counts -> ε-LDP randomized response -> tensor (N, D)
# -----------------------------
print("Building hashed+LDP features...")
X_train = build_hashed_features(df_train[cfg.text_col].tolist())
X_val   = build_hashed_features(df_val[cfg.text_col].tolist())
y_train = torch.tensor(df_train[label_col].values, dtype=torch.long)
y_val   = torch.tensor(df_val[label_col].values,   dtype=torch.long)

# -----------------------------
# Move data to device (CPU/GPU)
# -----------------------------
device = torch.device(cfg.device)
X_train, y_train = X_train.to(device), y_train.to(device)
X_val,   y_val   = X_val.to(device),   y_val.to(device)

# -----------------------------
# Initialize student model (router) and optimizer
# -----------------------------
model = StudentModel(cfg.d, cfg.emb_dim, cfg.hidden, cfg.dropout).to(device)
opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

# Track the best checkpoint by validation F1
best_f1 = -1.0

n = X_train.shape[0]
idx = torch.arange(n, device=device)

# -----------------------------
# Training loop (cross-entropy only for Day1 demo)
# Note: Full project later can add KD + COS losses for teacher alignment.
# -----------------------------
for ep in range(1, cfg.epochs + 1):
    model.train()

    # Shuffle indices each epoch
    perm = idx[torch.randperm(n)]
    total = 0.0

    # Mini-batch training
    for i in range(0, n, cfg.batch_size):
        b = perm[i:i + cfg.batch_size]
        xb, yb = X_train[b], y_train[b]

        opt.zero_grad()
        logits, _ = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        opt.step()

        total += loss.item() * xb.size(0)

    # -----------------------------
    # Validation metrics
    # -----------------------------
    f1, auc = evaluate(model, X_val, y_val)
    print(f"epoch {ep}: train_loss={total/n:.4f}  val_f1={f1:.4f}  val_auc={auc:.4f}")

    # -----------------------------
    # Save best checkpoint (by val F1)
    # Store metadata needed for reproducible inference
    # -----------------------------
    if f1 > best_f1:
        best_f1 = f1
        torch.save({
            "state_dict": model.state_dict(),
            "dim": cfg.d,
            "emb_dim": cfg.emb_dim,
            "epsilon": cfg.epsilon,
            "seed": cfg.seed,
            "text_col": cfg.text_col,
            "label_col": label_col
        }, cfg.out_path)
        print("✅ saved best ->", cfg.out_path)

print("best_val_f1 =", best_f1)

Using label_col: label
Building hashed+LDP features...
epoch 1: train_loss=0.6864  val_f1=0.7116  val_auc=0.7117
✅ saved best -> /content/drive/MyDrive/pphd_agent_demo/checkpoints/student_demo_D75_eps1.pt
epoch 2: train_loss=0.6501  val_f1=0.7294  val_auc=0.7194
✅ saved best -> /content/drive/MyDrive/pphd_agent_demo/checkpoints/student_demo_D75_eps1.pt
epoch 3: train_loss=0.6351  val_f1=0.7145  val_auc=0.7231
epoch 4: train_loss=0.6246  val_f1=0.6730  val_auc=0.7168
epoch 5: train_loss=0.6110  val_f1=0.6916  val_auc=0.7100
best_val_f1 = 0.7293844367015099


In [7]:
# -----------------------------
# Load the best student checkpoint (saved during training)
# -----------------------------
ckpt = torch.load(cfg.out_path, map_location=device)

# Reconstruct the exact same model shape used during training
# (dim = hashing dimension D, emb_dim = internal embedding size)
model2 = StudentModel(ckpt["dim"], ckpt["emb_dim"], cfg.hidden, cfg.dropout).to(device)
model2.load_state_dict(ckpt["state_dict"])
model2.eval()  # inference mode (disables dropout, etc.)


# -----------------------------
# Routing thresholds (probability -> risk tier)
# Note: thresholds can be tuned later based on desired operating points.
# -----------------------------
THRESH_MED = 0.636
THRESH_HIGH = 0.678

def risk_level(p: float) -> str:
    """
    Convert a stress probability into a discrete routing decision.
    """
    if p >= THRESH_HIGH:
        return "high"
    if p >= THRESH_MED:
        return "medium"
    return "low"


# -----------------------------
# Inference helper: raw text -> privatized vector -> model -> stress_prob + risk_level
# -----------------------------
@torch.no_grad()
def predict(text: str):
    """
    Predict stress probability from raw text, without exposing raw text downstream.

    Steps:
    1) Convert raw text to a D-dimensional privatized hashed vector (PPHD features)
    2) Run the student classifier to get logits
    3) Convert logits -> probability via softmax
    4) Map probability -> risk_level using routing thresholds
    """
    # Build privatized feature vector (shape: [1, D])
    x = torch.tensor(
        hashed_vector(text, cfg.d, cfg.epsilon, cfg.seed),
        dtype=torch.float32
    ).to(device).unsqueeze(0)

    # Forward pass
    logits, _ = model2(x)

    # Positive-class probability (stress=1)
    p = F.softmax(logits, dim=-1)[0, 1].item()

    return p, risk_level(p)


# -----------------------------
# Demo input (raw text stays local)
# -----------------------------
text = "I feel anxious and can't sleep for 2 weeks. Sometimes I feel hopeless."
p, lvl = predict(text)


# -----------------------------
# Build a privacy-safe payload for downstream components (NO raw text)
# - risk_level / stress_prob: routing signal
# - slots: structured summary for retrieval (Day1 uses placeholder; later replaced by extract_slots)
# - privacy: metadata for auditability
# -----------------------------
payload = {
    "risk_level": lvl,
    "stress_prob": round(p, 4),
    "slots": {  # Day1 placeholder; later replace with: payload["slots"] = extract_slots(text)
        "symptoms": ["anxiety", "sleep_disturbance"],
        "duration": "2_weeks",
        "severity": None,
        "safety_flags": []
    },
    "privacy": {"epsilon": cfg.epsilon, "hash_dim": cfg.d, "seed": cfg.seed},
    "version": {"student": "student_demo_D75_eps1", "policy": "v0.1"},
}

print(json.dumps(payload, indent=2))

{
  "risk_level": "high",
  "stress_prob": 0.7181,
  "slots": {
    "symptoms": [
      "anxiety",
      "sleep_disturbance"
    ],
    "duration": "2_weeks",
    "severity": null,
    "safety_flags": []
  },
  "privacy": {
    "epsilon": 1.0,
    "hash_dim": 75,
    "seed": 0
  },
  "version": {
    "student": "student_demo_D75_eps1",
    "policy": "v0.1"
  }
}


In [8]:
# -----------------------------
# Audit logging (privacy-safe)
# -----------------------------
# We persist ONLY the routing payload + metadata for reproducibility and monitoring.
# IMPORTANT: Do NOT log raw user text here (PHI/PII must not be written to disk).

from datetime import datetime, timezone

AUDIT_PATH = os.path.join(PROJECT_DIR, "audit_log.jsonl")

# One JSONL record per routing decision
event = {
    "ts": datetime.now(timezone.utc).isoformat(),  # timezone-aware UTC timestamp
    "event": "router_output",                      # event type (useful for filtering/analytics)
    "payload": payload                             # privacy-safe payload (no raw text)
}

# Append to JSONL (audit-friendly format: one JSON object per line)
with open(AUDIT_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(event) + "\n")

print("audit ->", AUDIT_PATH)

audit -> /content/drive/MyDrive/pphd_agent_demo/audit_log.jsonl


In [9]:
# -----------------------------
# Slot extraction (local-only, rule-based)
# -----------------------------
# Purpose:
# - Extract a minimal, de-identified structured summary ("slots") from the raw text
# - Slots are used downstream for retrieval (RAG) without sending raw text off-device
# - This is a lightweight Day1 baseline (fast + explainable). It can be upgraded later.

import re

# Keyword dictionaries for simple symptom detection (demo baseline)
SYMPTOMS = {
  "anxiety": ["anxiety", "anxious", "panic", "worried", "worry"],
  "sleep_disturbance": ["insomnia", "can't sleep", "cant sleep", "trouble sleeping", "sleep"],
  "depression": ["depressed", "hopeless", "empty", "depression", "worthless"]
}

# Regex patterns for duration and safety flags
DURATION_PAT = re.compile(r"(\b\d+\b)\s*(day|days|week|weeks|month|months|year|years)")
SELF_HARM_PAT = [
    r"\bsuicid(e|al)\b",
    r"\bkill myself\b",
    r"\bself[- ]harm\b",
    r"\bend it\b"
]

def extract_slots(text: str):
    """
    Extract a small set of structured fields from the raw text.

    Returns:
        {
          "symptoms": [<symptom_categories>],
          "duration": "<N>_<unit>" or None,
          "severity": None (placeholder for future extension),
          "safety_flags": ["self_harm_risk"] if triggered
        }

    Notes:
    - Runs locally on raw text.
    - Output should avoid PHI/PII. We only return coarse categories and relative duration.
    """
    t = text.lower()

    # 1) Symptom category detection via keyword matching
    symptoms = []
    for slot, kws in SYMPTOMS.items():
        if any(kw in t for kw in kws):
            symptoms.append(slot)

    # 2) Duration extraction (e.g., "2 weeks" -> "2_weeks")
    dur = None
    m = DURATION_PAT.search(t)
    if m:
        dur = f"{m.group(1)}_{m.group(2)}"

    # 3) Safety flags (e.g., self-harm language)
    safety_flags = []
    if any(re.search(p, t) for p in SELF_HARM_PAT):
        safety_flags.append("self_harm_risk")

    return {
        "symptoms": symptoms[:5],
        "duration": dur,
        "severity": None,
        "safety_flags": safety_flags
    }

In [10]:
payload["slots"] = extract_slots(text)
print(payload["slots"])

{'symptoms': ['anxiety', 'sleep_disturbance', 'depression'], 'duration': '2_week', 'severity': None, 'safety_flags': []}


In [11]:
# -----------------------------
# Persist a sample payload to Drive (for reuse)
# -----------------------------
PAYLOAD_PATH = os.path.join(PROJECT_DIR, "sample_payload.json")

with open(PAYLOAD_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("saved ->", PAYLOAD_PATH)

saved -> /content/drive/MyDrive/pphd_agent_demo/sample_payload.json
